# PHASE 1: WEB SCRAPING

### Importing Libraries

In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re

### Base URL + Setup

In [3]:
base_url = "http://books.toscrape.com/catalogue/page-{}.html"

data = []

headers = {
    "User-Agent": "Mozilla/5.0"
}

# Rating mapping
rating_map = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

### Testing Single Page

In [4]:
url = base_url.format(1)

response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.text, "html.parser")

books = soup.find_all("article", class_="product_pod")

print("Books found:", len(books))

Books found: 20


### Extract Data from One Page

In [5]:
for book in books:
    
    # Title
    title = book.h3.a["title"]

    # Price (FIXED)
    price = book.find("p", class_="price_color").text
    price = float(re.sub(r"[^\d.]", "", price))

    # Rating
    rating_class = book.p["class"][1]
    rating = rating_map.get(rating_class, 0)

    # Availability
    availability = book.find("p", class_="instock availability").text.strip()

    # Product URL
    link = book.h3.a["href"]
    product_url = "http://books.toscrape.com/catalogue/" + link

    data.append([title, price, rating, availability, product_url])

# Preview
print(data[:5])

[['A Light in the Attic', 51.77, 3, 'In stock', 'http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html'], ['Tipping the Velvet', 53.74, 1, 'In stock', 'http://books.toscrape.com/catalogue/tipping-the-velvet_999/index.html'], ['Soumission', 50.1, 1, 'In stock', 'http://books.toscrape.com/catalogue/soumission_998/index.html'], ['Sharp Objects', 47.82, 4, 'In stock', 'http://books.toscrape.com/catalogue/sharp-objects_997/index.html'], ['Sapiens: A Brief History of Humankind', 54.23, 5, 'In stock', 'http://books.toscrape.com/catalogue/sapiens-a-brief-history-of-humankind_996/index.html']]


### Full Loop (All Pages)

In [6]:
data = []  # reset before full run

for page in range(1, 51):
    print(f"Scraping page {page}...")

    url = base_url.format(page)
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")

    books = soup.find_all("article", class_="product_pod")

    for book in books:
        
        title = book.h3.a["title"]

        price = book.find("p", class_="price_color").text
        price = float(re.sub(r"[^\d.]", "", price))

        rating_class = book.p["class"][1]
        rating = rating_map.get(rating_class, 0)

        availability = book.find("p", class_="instock availability").text.strip()

        link = book.h3.a["href"]
        product_url = "http://books.toscrape.com/catalogue/" + link

        data.append([title, price, rating, availability, product_url])

    time.sleep(1)  # avoid hitting server too fast

Scraping page 1...
Scraping page 2...
Scraping page 3...
Scraping page 4...
Scraping page 5...
Scraping page 6...
Scraping page 7...
Scraping page 8...
Scraping page 9...
Scraping page 10...
Scraping page 11...
Scraping page 12...
Scraping page 13...
Scraping page 14...
Scraping page 15...
Scraping page 16...
Scraping page 17...
Scraping page 18...
Scraping page 19...
Scraping page 20...
Scraping page 21...
Scraping page 22...
Scraping page 23...
Scraping page 24...
Scraping page 25...
Scraping page 26...
Scraping page 27...
Scraping page 28...
Scraping page 29...
Scraping page 30...
Scraping page 31...
Scraping page 32...
Scraping page 33...
Scraping page 34...
Scraping page 35...
Scraping page 36...
Scraping page 37...
Scraping page 38...
Scraping page 39...
Scraping page 40...
Scraping page 41...
Scraping page 42...
Scraping page 43...
Scraping page 44...
Scraping page 45...
Scraping page 46...
Scraping page 47...
Scraping page 48...
Scraping page 49...
Scraping page 50...


### Creating DataFrame

In [7]:
df = pd.DataFrame(data, columns=[
    "Title", "Price", "Rating", "Availability", "Product_URL"
])

df.head()

,Title,Price,Rating,Availability,Product_URL
0,A Light in the Attic,51.77,3,In stock,http://books.toscrape.com/catalogue/a-light-in...
1,Tipping the Velvet,53.74,1,In stock,http://books.toscrape.com/catalogue/tipping-th...
2,Soumission,50.10,1,In stock,http://books.toscrape.com/catalogue/soumission...
3,Sharp Objects,47.82,4,In stock,http://books.toscrape.com/catalogue/sharp-obje...
4,Sapiens: A Brief History of Humankind,54.23,5,In stock,http://books.toscrape.com/catalogue/sapiens-a-...


### Validate Data

In [8]:
print("Shape:", df.shape)
print(df.info())

Shape: (1000, 5)
<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Title         1000 non-null   str    
 1   Price         1000 non-null   float64
 2   Rating        1000 non-null   int64  
 3   Availability  1000 non-null   str    
 4   Product_URL   1000 non-null   str    
dtypes: float64(1), int64(1), str(3)
memory usage: 39.2 KB
None


### Save CSV

In [9]:
df.to_csv("raw_books_data.csv", index=False)
print("File saved successfully!")

File saved successfully!


## Extracting & Adding Category column by visiting each product page

### Loading Scraped Data

In [10]:
import pandas as pd

df = pd.read_csv("raw_books_data.csv")

df.head()

,Title,Price,Rating,Availability,Product_URL
0,A Light in the Attic,51.77,3,In stock,http://books.toscrape.com/catalogue/a-light-in...
1,Tipping the Velvet,53.74,1,In stock,http://books.toscrape.com/catalogue/tipping-th...
2,Soumission,50.10,1,In stock,http://books.toscrape.com/catalogue/soumission...
3,Sharp Objects,47.82,4,In stock,http://books.toscrape.com/catalogue/sharp-obje...
4,Sapiens: A Brief History of Humankind,54.23,5,In stock,http://books.toscrape.com/catalogue/sapiens-a-...


### Importing Required Libraries

In [11]:
import requests
from bs4 import BeautifulSoup
import time

headers = {
    "User-Agent": "Mozilla/5.0"
}

### Testing Category Extraction
### Testing on 1 URL before full run

In [12]:
url = df["Product_URL"][0]

response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.text, "html.parser")

# Extract category from breadcrumb
breadcrumb = soup.find("ul", class_="breadcrumb").find_all("li")

category = breadcrumb[2].text.strip()

print("Category:", category)

Category: Poetry


### Creating Function

In [13]:
def get_category(url):
    try:
        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.text, "html.parser")
        
        breadcrumb = soup.find("ul", class_="breadcrumb").find_all("li")
        category = breadcrumb[2].text.strip()
        
        return category
    
    except:
        return "Unknown"

### Testing on Few Rows

In [14]:
df_sample = df.head(10).copy()

df_sample["Category"] = df_sample["Product_URL"].apply(get_category)

df_sample

,Title,Price,Rating,Availability,Product_URL,Category
0,A Light in the Attic,51.77,3,In stock,http://books.toscrape.com/catalogue/a-light-in...,Poetry
1,Tipping the Velvet,53.74,1,In stock,http://books.toscrape.com/catalogue/tipping-th...,Historical Fiction
2,Soumission,50.10,1,In stock,http://books.toscrape.com/catalogue/soumission...,Fiction
3,Sharp Objects,47.82,4,In stock,http://books.toscrape.com/catalogue/sharp-obje...,Mystery
4,Sapiens: A Brief History of Humankind,54.23,5,In stock,http://books.toscrape.com/catalogue/sapiens-a-...,History
5,The Requiem Red,22.65,1,In stock,http://books.toscrape.com/catalogue/the-requie...,Young Adult
6,The Dirty Little Secrets of Getting Your Dream...,33.34,4,In stock,http://books.toscrape.com/catalogue/the-dirty-...,Business
7,The Coming Woman: A Novel Based on the Life of...,17.93,3,In stock,http://books.toscrape.com/catalogue/the-coming...,Default
8,The Boys in the Boat: Nine Americans and Their...,22.60,4,In stock,http://books.toscrape.com/catalogue/the-boys-i...,Default
9,The Black Maria,52.15,1,In stock,http://books.toscrape.com/catalogue/the-black-...,Poetry


### Applying to Full Dataset

In [15]:
categories = []

for i, url in enumerate(df["Product_URL"]):
    print(f"Processing {i+1}/{len(df)}")

    category = get_category(url)
    categories.append(category)

    time.sleep(1)  # VERY IMPORTANT (avoid blocking)

df["Category"] = categories

Processing 1/1000
Processing 2/1000
Processing 3/1000
Processing 4/1000
Processing 5/1000
Processing 6/1000
Processing 7/1000
Processing 8/1000
Processing 9/1000
Processing 10/1000
Processing 11/1000
Processing 12/1000
Processing 13/1000
Processing 14/1000
Processing 15/1000
Processing 16/1000
Processing 17/1000
Processing 18/1000
Processing 19/1000
Processing 20/1000
Processing 21/1000
Processing 22/1000
Processing 23/1000
Processing 24/1000
Processing 25/1000
Processing 26/1000
Processing 27/1000
Processing 28/1000
Processing 29/1000
Processing 30/1000
Processing 31/1000
Processing 32/1000
Processing 33/1000
Processing 34/1000
Processing 35/1000
Processing 36/1000
Processing 37/1000
Processing 38/1000
Processing 39/1000
Processing 40/1000
Processing 41/1000
Processing 42/1000
Processing 43/1000
Processing 44/1000
Processing 45/1000
Processing 46/1000
Processing 47/1000
Processing 48/1000
Processing 49/1000
Processing 50/1000
Processing 51/1000
Processing 52/1000
Processing 53/1000
Pr

### Checking Results

In [16]:
df.head()

print(df["Category"].value_counts())

Category
Default               152
Nonfiction            110
Sequential Art         75
Add a comment          67
Fiction                65
Young Adult            54
Fantasy                48
Romance                35
Mystery                32
Food and Drink         30
Childrens              29
Historical Fiction     26
Poetry                 19
Classics               19
History                18
Horror                 17
Womens Fiction         17
Science Fiction        16
Science                14
Music                  13
Business               12
Travel                 11
Thriller               11
Philosophy             11
Humor                  10
Autobiography           9
Art                     8
Religion                7
Psychology              7
Spirituality            6
New Adult               6
Christian Fiction       6
Self Help               5
Biography               5
Sports and Games        5
Health                  4
Politics                3
Contemporary            3
Chr

### Saving Updated Dataset

In [17]:
df.to_csv("books_with_category.csv", index=False)

print("Category added successfully!")

Category added successfully!


In [19]:
df.head(1000)

,Title,Price,Rating,Availability,Product_URL,Category
0,A Light in the Attic,51.77,3,In stock,http://books.toscrape.com/catalogue/a-light-in...,Poetry
1,Tipping the Velvet,53.74,1,In stock,http://books.toscrape.com/catalogue/tipping-th...,Historical Fiction
2,Soumission,50.10,1,In stock,http://books.toscrape.com/catalogue/soumission...,Fiction
3,Sharp Objects,47.82,4,In stock,http://books.toscrape.com/catalogue/sharp-obje...,Mystery
4,Sapiens: A Brief History of Humankind,54.23,5,In stock,http://books.toscrape.com/catalogue/sapiens-a-...,History
...,...,...,...,...,...,...
995,Alice in Wonderland (Alice's Adventures in Won...,55.53,1,In stock,http://books.toscrape.com/catalogue/alice-in-w...,Classics
996,"Ajin: Demi-Human, Volume 1 (Ajin: Demi-Human #1)",57.06,4,In stock,http://books.toscrape.com/catalogue/ajin-demi-...,Sequential Art
997,A Spy's Devotion (The Regency Spies of London #1),16.97,5,In stock,http://books.toscrape.com/catalogue/a-spys-dev...,Historical Fiction
998,1st to Die (Women's Murder Club #1),53.98,1,In stock,http://books.toscrape.com/catalogue/1st-to-die...,Mystery
